<a href="https://colab.research.google.com/github/elijahschoeniger-cpu/DS2002FA26/blob/main/2026_09_25_%E2%80%94_Cleaning_Gauntlet_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
log('dedupe', 'dropped exact duplicate rows', before - len(df))

[dedupe] dropped exact duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [4]:
# TODO
before_dtype = df['price'].dtype
df['price'] = pd.to_numeric(df['price'].str.replace('$', '', regex=False), errors='coerce')
n_bad = df['price'].isna().sum()
log('price', f'stripped "$", cast {before_dtype} -> {df["price"].dtype}; {n_bad} unparseable', len(df))

[price] stripped "$", cast object -> float64; 0 unparseable (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
# TODO
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')
before = len(df)
bad = df['qty'].isna() | (df['qty'] < 0)
df = df.loc[~bad].copy()
df['qty'] = df['qty'].astype(int)
log('qty', 'coerced to numeric, dropped rows with NaN or negative qty', before - len(df))

[qty] coerced to numeric, dropped rows with NaN or negative qty (25 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse
print(df['item'].value_counts())

ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger':  'Foam Finger',
    'foam finger':  'Foam Finger',
    'Rain Poncho':  'Rain Poncho',
    'rain poncho':  'Rain Poncho',
}

unmapped = set(df['item']) - set(ITEM_MAP)
assert not unmapped, f'unmapped item spellings: {unmapped}'

before = df['item'].nunique()
df['item'] = df['item'].map(ITEM_MAP)
log('item', f'mapped {before} spellings -> {df["item"].nunique()} canonical products', len(df))

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[item] mapped 6 spellings -> 3 canonical products (275 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
# TODO
print(df['category'].value_counts())

CATEGORY_MAP = {
    'Food':      'Food',
    'food':      'Food',
    'Merch':     'Merch',
    'Apparel':   'Merch',      # business decision: Apparel is a subset of Merch, collapse it
    'RainGear':  'Rain Gear',
    'rain-gear': 'Rain Gear',
}

unmapped = set(df['category']) - set(CATEGORY_MAP)
assert not unmapped, f'unmapped category spellings: {unmapped}'

n_apparel = (df['category'] == 'Apparel').sum()
before = df['category'].nunique()
df['category'] = df['category'].map(CATEGORY_MAP)
log('category',
    f'mapped {before} spellings -> {df["category"].nunique()}; folded Apparel into Merch (business call)',
    n_apparel)

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[category] mapped 6 spellings -> 3; folded Apparel into Merch (business call) (43 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [12]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert set(df['item']) == {'Cheeseburger', 'Foam Finger', 'Rain Poncho'}
assert set(df['category']) == {'Food', 'Merch', 'Rain Gear'}
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [13]:
# TODO
df['revenue'] = (df['qty'] * df['price']).round(2)

by_cat = (df.groupby('category')['revenue']
            .sum()
            .round(2)
            .sort_values(ascending=False))
print(by_cat.to_string())
print(f'\nTOTAL: {df["revenue"].sum():.2f}')

category
Food         1656.0
Merch        1572.0
Rain Gear    1512.0

TOTAL: 4740.00


**What I would tell the vendor:** _..._

### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,dedupe,dropped exact duplicate rows,15
1,price,"stripped ""$"", cast object -> float64; 0 unpars...",300
2,qty,"coerced to numeric, dropped rows with NaN or n...",25
3,item,mapped 6 spellings -> 3 canonical products,275
4,category,mapped 6 spellings -> 3; folded Apparel into M...,43


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

_your answer here_